In [ ]:
import numpy as np
import sysl.symbolic as sls
import geolipi.symbolic as gls
from sysl.shader.evaluate import evaluate_to_shader
from sysl.shader_runtime.generate_shader_html import create_shader_html, make_jupyter_compatible_html
from sysl.shader_runtime.generate_shader_html import create_multibuffer_shader_html
from IPython.display import display, HTML

settings = {
    "render_mode": "v4",
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": False,
        "_AA": 2,
        "_RAYCAST_MAX_STEPS": 200,
    },
    "set_to_ubo": False,
    "export_params": False,
}



In [ ]:
# All 3D primitives expressions
all_primitives_expressions = [
    gls.Sphere3D((0.5,)),
    gls.Box3D((0.3, 0.1, 0.2,)),
    gls.Cuboid3D((0.3, 0.1, 0.2,)),
    gls.RoundedBox3D((0.3, 0.1, 0.2,), (0.1,)),
    gls.BoxFrame3D((0.3, 0.5, 0.6,), (0.1,)),
    gls.Torus3D((0.5, 0.1,)),
    gls.CappedTorus3D((0.5,), (0.5,), (0.21,)),
    gls.Link3D(0.5, 0.3, 0.1),
    gls.InfiniteCylinder3D((0.3, 0.1, 0.2,)),
    gls.InfiniteCone3D((0.5,)),
    gls.Cone3D((0.5,), (0.5,)),
    gls.InexactCone3D((0.5,), (0.5,)),
    gls.Plane3D((0.3, 0.1, 0.2,), (0.5,)),
    gls.HexPrism3D((0.3, 0.1,)),
    gls.TriPrism3D((0.3, 0.1,)),
    gls.Capsule3D((0.3, 0.1, 0.2,), (0.3, 0.6, 0.2,), (0.1,)),
    gls.VerticalCapsule3D((0.5,), (0.1,)),
    gls.CappedCylinder3D((0.5,), (0.1,)),
    gls.Cylinder3D((0.5,), (0.1,)),
    gls.ArbitraryCappedCylinder3D((0.3, 0.1, 0.2,), (0.8, 0.1, 0.6,), (0.1,)),
    gls.RoundedCylinder3D((0.1,), (0.5,), (0.4,)),
    gls.CappedCone3D((0.1,), (0.3,), (0.4,)),
    gls.ArbitraryCappedCone3D((0.3, 0.1, 0.2,), (0.3, 0.6, 0.2,), (0.3,), (0.1,)),
    gls.SolidAngle3D((0.5,), (0.3,)),
    gls.CutSphere3D((0.5,), (0.2,)),
    gls.CutHollowSphere((0.3,), (0.5,), (0.1,)),
    gls.DeathStar3D((0.5,), (0.35,), (0.5,)),
    gls.RoundCone3D((0.3,), (0.1,), (0.5,)),
    gls.ArbitraryRoundCone3D((0.1, 0.1, 0.2,), (0.3, 0.7, 0.8,), (0.3,), (0.1,)),
    gls.InexactEllipsoid3D((0.3, 0.1, 0.2,)),
    gls.RevolvedVesica3D((0.0, 0.1, 0.2,), (0.3, 0.1, 0.6,), (0.1,)),
    gls.Rhombus3D((0.3,), (0.1,), (0.5,), (0.1,)),
    gls.Octahedron3D((0.5,)),
    gls.InexactOctahedron3D((0.5,)),
    gls.Pyramid3D((0.5,)),
    # Here on its tricky.
    gls.Triangle3D((0.3, 0.1, 0.2,), (0.3, 0.4, 0.2,), (0.1, 0.0, 0.6,)),
    gls.Quadrilateral3D((0.0, 0.1, 0.2,), (0.3, 0.4, 0.2,), (0.1, 0.0, 0.6,), (0.7, 0.1, 0.2,)),
    gls.NoParamCuboid3D(),
    gls.NoParamSphere3D(),
    gls.NoParamCylinder3D(),
    # Not really usable.
    # gls.PlaneV23D((0.3, 0.1, 0.2,), (0.3, 0.1, 0.2,)),
    # gls.VerticalCappedCylinder3D((0.5,), (0.1,)),
    # gls.InexactSuperQuadrics3D((0.3, 0.1, 0.2,), (0.3,), (0.1,)),
    # gls.InexactAnisotropicGaussian3D((0.0, 0.0, 0.0,), (0.1, 0.1, 0.1,), (0.1,)),
    # gls.NullExpression3D(),
]


In [ ]:
SEL_INDEX = 0
cur_expr = all_primitives_expressions[SEL_INDEX]
# cur_expr = gls.Dilate3D(cur_expr, (0.05,))
print(cur_expr)
material = sls.MatRefV4("MatPlastic")
scene_with_material = sls.MatSolid(cur_expr, material)

# Get Shader Code
shader_info = evaluate_to_shader(scene_with_material, settings=settings, 
                                 mode="multipass", post_process_shader=["all_outline_nobg"])
# TO visualize in a browser:
html_code = create_multibuffer_shader_html(shader_info, show_controls=True)
# To visualize inline in jupyter notebook:
with open("/users/aganesh8/data/aganesh8/projects/project_neo/old/new_renders/html/test.html", "w") as f:
    f.write(html_code)
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
# display(HTML(jupy_wrapper_html))

In [ ]:
# Also check match between eval in render. 
from sysl.shader.utils.texture import recursive_encode_texture_tensor
from geolipi.torch_compute import Sketcher, recursive_evaluate
sketcher = Sketcher(n_dims=3, resolution=64)

# cur_expr = gls.ArbitraryCappedCylinder3D((0.3, 0.0, 0.1, ), (0.8, 0.0, 0.7,), (0.1,))
eval_out = recursive_evaluate(cur_expr.tensor(), sketcher)# [..., 0]
if len(eval_out.shape) == 2:
    eval_out = eval_out[..., 0]
expr = gls.SDFGrid3D(eval_out, "torch_eval", (0.2,))

expr = recursive_encode_texture_tensor(expr, sketcher)

new_expr = sls.MatSolidV1(expr, material)

final_scene = gls.Union(
    new_expr,
    gls.Translate3D(scene_with_material, (1.0, 0.0, 0.0))
)


shader_code, uniforms, textures = evaluate_to_shader(final_scene, settings=settings)
# TO visualize in a browser:
with open("shader_code.glsl", "w") as f:
    f.write(shader_code)
html_code = create_shader_html(shader_code, uniforms, textures, show_controls=False)
# To visualize inline in jupyter notebook:
with open("test.html", "w") as f:
    f.write(html_code)
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
# display(HTML(jupy_wrapper_html))
